# 時代別の単語ベクトルを位置合わせして意味が変わった単語を探す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_lexical_semantics/semantic_change_w2v.ipynb)

NLP クックブック 18-10 ／ 語の意味と意味の変化

時代によって意味や使われ方が変わった単語を、単語ベクトルを使って自動的に見つけたい。

In [1]:
!pip install -q datasets fugashi unidic-lite gensim

In [2]:
import random, re
import numpy as np
from datasets import load_dataset
from fugashi import Tagger
from gensim.models import Word2Vec

ds = load_dataset("globis-university/aozorabunko-clean", split="train")
tagger = Tagger()
random.seed(0)

def era(meta):
    year = (meta.get("生年月日") or "")[:4]
    if meta["文字遣い種別"] != "新字新仮名" or not year.isdigit():
        return None
    return "旧" if 1850 <= int(year) < 1880 else "新" if 1895 <= int(year) < 1930 else None

texts, common_nouns = {"旧": [], "新": []}, set()
for ex in ds:
    if era(ex["meta"]):
        texts[era(ex["meta"])].append(ex["text"])

def tokenize(works, max_chars=20_000_000):
    sentences, n = [], 0
    for text in random.sample(works, len(works)):
        for s in re.split(r"(?<=[。！？])|\n", text[:30000]):
            if len(s.strip()) >= 5:
                words = tagger(s.strip())
                common_nouns.update(w.surface for w in words if w.feature.pos2 == "普通名詞")
                sentences.append([w.surface for w in words]); n += len(s)
        if n > max_chars:
            break
    return sentences

corpus = {e: tokenize(t) for e, t in texts.items()}
print({e: len(s) for e, s in corpus.items()})

{'旧': 489773, '新': 557360}


In [3]:
# 時代ごとに別々の word2vec を学習する (それぞれ 1 分ほど)
models = {e: Word2Vec(s, vector_size=100, window=5, min_count=30, sg=1, epochs=3, workers=4, seed=0).wv for e, s in corpus.items()}

# 2 つの空間は向きがばらばらなので、そのままでは比べられない。両方に十分出てくる語を使って、回転で重ね合わせる (直交プロクラステス)
vocab = [w for w in models["旧"].index_to_key if w in models["新"].key_to_index
         and models["旧"].get_vecattr(w, "count") >= 100 and models["新"].get_vecattr(w, "count") >= 100]
A = np.stack([models["旧"][w] for w in vocab]); A /= np.linalg.norm(A, axis=1, keepdims=True)
B = np.stack([models["新"][w] for w in vocab]); B /= np.linalg.norm(B, axis=1, keepdims=True)
U, _, Vt = np.linalg.svd(A.T @ B)
R = U @ Vt                                         # A @ R が B に最も近くなる回転行列
similarity = dict(zip(vocab, ((A @ R) * B).sum(axis=1)))
print(len(vocab), "語 / 位置合わせ前の平均コサイン", round(float((A * B).sum(axis=1).mean()), 3), "→ 後", round(float(np.mean(list(similarity.values()))), 3))

5408 語 / 位置合わせ前の平均コサイン 0.219 → 後 0.788


In [4]:
# 重ね合わせた後もベクトルが離れている語 = 使われ方が変わった語
targets = [w for w in vocab if w in common_nouns and re.fullmatch(r"[一-龥ァ-ヶー]{2,}", w)]
ranked = sorted(targets, key=similarity.get)
print("変化が大きい:", [(w, round(float(similarity[w]), 2)) for w in ranked[:30]])
print("変化が小さい:", [(w, round(float(similarity[w]), 2)) for w in ranked[-15:]])

for w in ["原子", "同志", "活動", "戦争", "自由", "映画", "言葉"]:
    old = [x for x, _ in models["旧"].most_similar(w, topn=6)]
    new = [x for x, _ in models["新"].most_similar(w, topn=6)]
    print(f"{w} ({similarity[w]:.2f})  旧: {' '.join(old)}  →  新: {' '.join(new)}")

変化が大きい: [('深山', 0.41), ('法師', 0.43), ('バック', 0.43), ('名人', 0.44), ('手先', 0.44), ('先達', 0.47), ('甲斐', 0.48), ('山川', 0.48), ('妻君', 0.5), ('浮浪', 0.52), ('川上', 0.53), ('島田', 0.53), ('子爵', 0.53), ('登場', 0.54), ('五郎', 0.54), ('屍体', 0.55), ('アン', 0.55), ('学士', 0.56), ('レコード', 0.56), ('農民', 0.57), ('公爵', 0.57), ('信者', 0.57), ('部落', 0.57), ('シナ', 0.58), ('退屈', 0.58), ('漁夫', 0.58), ('ホール', 0.58), ('平野', 0.58), ('三郎', 0.58), ('文字', 0.58)]
変化が小さい: [('カ月', 0.9), ('引用', 0.9), ('推移', 0.9), ('中庭', 0.9), ('初夏', 0.9), ('女中', 0.9), ('尊重', 0.9), ('出身', 0.9), ('接触', 0.91), ('水面', 0.91), ('表紙', 0.91), ('考察', 0.91), ('物音', 0.91), ('両側', 0.91), ('根柢', 0.91)]
原子 (0.73)  旧: 質量 地殻 電子 生成 エントロピー 弾性  →  新: 核 爆弾 兵器 水素 衛星 エネルギー
同志 (0.63)  旧: 朋友 妻女 同窓 父兄 門弟 交わり  →  新: 党員 闘士 兄弟 姉妹 同胞 ノラ
活動 (0.72)  旧: 演技 改良 適応 開展 影像 色盲  →  新: サークル 分裂 擁護 全面 営利 一員
戦争 (0.71)  旧: 日露 日清 革命 事変 震災 大戦  →  新: 敗戦 侵略 日露 ファシズム 大戦 軍隊
自由 (0.78)  旧: 規律 公平 健全 独立 拘束 誠実  →  新: 安定 如意 合理 言論 隷属 拘束
映画 (0.73)  旧: 連句 トーキー レビュー 発声 モンタージュ 場面  →  新: 演劇 劇 絵画 メイエルホ

- 時代ごとに単語ベクトルを学習し、回転で重ね合わせてから同じ単語のベクトルを比べる。コサイン類似度が低い語ほど、周りに出てくる語（文脈）が変わっている。
- 「原子」（物理の用語 → 爆弾）、「同志」（仲間一般 → 政治運動の仲間）のように、近い語の顔ぶれを並べると、何が変わったかが読み取れる。数値だけでなく、必ず近傍語を確認する。
- 上位には、人名と同じ形の語や、特定の作品に偏って出てくる語も混ざる。本当の意味変化と、コーパスの偏りを区別するのは人の仕事になる。
- 頻度の低い語はベクトルが不安定で、それだけで「変化が大きい」と判定されやすい。頻度の下限を設け、できれば乱数シードを変えて結果が安定しているか確かめる。
- この方法は Hamilton らの手法（SGNS + 直交プロクラステス）で、意味変化の検出の標準的なベースライン。
- 同じ直交プロクラステスで 2 言語の単語ベクトルを重ね、対訳語を引く例は、単語ベクトルの章の「2 言語の単語ベクトルを線形写像で対応させて対訳語を引く」を参照。